<a href="https://colab.research.google.com/github/hilalozkan22/ConfidenceCalibration/blob/main/turkish_llm_confidence_pilot_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Türkçe Açık Uçlu QA Güven Kalibrasyonu — Pilot v2

Bu notebook, küçük ve quantized bir açık kaynak LLM üzerinde Türkçe açık uçlu olgusal soru-cevap deneyini uçtan uca çalıştırır.

## Amaç

Her model cevabı için birden fazla ham güven ve belirsizlik sinyali toplamak:

- Token log-probability
- Token entropy
- Sayısal self-reported confidence
- A/B/C self-verification olasılıkları
- Self-consistency ve örnekleme çeşitliliği
- `Bilmiyorum` / refusal göstergeleri
- Cevap uzunluğu ve format bilgileri

Daha sonra bu sinyaller, cevabın doğru olma olasılığını tahmin eden post-hoc bir model için feature olarak hazırlanır:

\[
\hat{p}=P(y=1\mid X)
\]

Burada `y = manual_is_correct`.

## Pilot v2 değişiklikleri

- Soru seti 20 sorudan **60 soruya** çıkarıldı.
- Kolay/orta/zor etiketi zorunlu olmaktan çıkarıldı.
- Token entropy feature'ları eklendi.
- Self-consistency doğrudan ana pipeline'a dahil edildi.
- Verification margin ve entropy eklendi.
- Unknown/refusal ve sample-level unknown oranları eklendi.
- Ground-truth leakage içeren alanlar model feature tablosundan ayrıldı.
- Her soru sonrasında checkpoint CSV kaydı eklendi.
- Manuel etiketleme ve post-hoc feature matrisi için ayrı aşamalar eklendi.

> **Önemli:** `automatic_alias_match`, referans cevap, alias listesi ve sampled cevapların doğruluk oranı post-hoc model feature'ı değildir. Bunlar ground-truth bilgisi taşır.

# Aşama 1 — GPU ve çalışma ortamı kontrolü

In [1]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU etkin değil. Colab çalışma ortamını T4 GPU olarak ayarlayın.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU belleği:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA kullanılabilir: True
GPU: Tesla T4
GPU belleği: 14.56 GB


# Aşama 2 — Gerekli paketlerin kurulması

Colab'ın hazır `numpy` ve `pandas` sürümlerini yükseltmiyoruz. Böylece Colab, cuDF ve numba bağımlılık çakışmalarını önlüyoruz.

Kurulumdan sonra Colab yeniden başlatma uyarısı verirse oturumu yeniden başlatıp Aşama 1'den devam edin.

In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

# Aşama 3 — İçe aktarmalar ve deney konfigürasyonu

In [3]:
import gc
import json
import math
import os
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3-4B"
QUANTIZATION_NAME = "4bit_nf4"
PIPELINE_VERSION = "open_qa_v2"

OUTPUT_DIR = Path("/content")
RAW_RESULTS_PATH = OUTPUT_DIR / "open_qa_pilot_60_raw_v2.csv"
ANNOTATION_TEMPLATE_PATH = OUTPUT_DIR / "open_qa_pilot_60_annotation_template_v2.csv"
ANNOTATED_RESULTS_PATH = OUTPUT_DIR / "open_qa_pilot_60_annotated_v2.csv"
MODEL_FEATURES_PATH = OUTPUT_DIR / "open_qa_pilot_60_model_features_v2.csv"

ANSWER_MAX_NEW_TOKENS = 20
CONFIDENCE_MAX_NEW_TOKENS = 10
VERIFICATION_MAX_NEW_TOKENS = 5

SELF_CONSISTENCY_K = 5
SELF_CONSISTENCY_SEEDS = [101, 202, 303, 404, 505]
SAMPLE_TEMPERATURE = 0.7
SAMPLE_TOP_P = 0.9

RUN_SELF_CONSISTENCY = True
RESUME_FROM_CHECKPOINT = True

print("Model:", MODEL_ID)
print("Sonuç dizini:", OUTPUT_DIR)
print("Self-consistency K:", SELF_CONSISTENCY_K)

Model: Qwen/Qwen3-4B
Sonuç dizini: /content
Self-consistency K: 5


# Aşama 4 — Model ve tokenizer yükleme

In [4]:
gc.collect()
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("Model yükleniyor...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

print("\nModel başarıyla yüklendi.")
print("Model cihazı:", model.device)
print("EOS:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("PAD:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("Chat template mevcut:", tokenizer.chat_template is not None)
print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Tokenizer yükleniyor...


Model yükleniyor...


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model başarıyla yüklendi.
Model cihazı: cuda:0
EOS: '<|im_end|>' 151645
PAD: '<|endoftext|>' 151643
Chat template mevcut: True
GPU allocated: 2.49 GB


# Aşama 5 — Prompt şablonları

Ana cevap üretiminde chain-of-thought kullanılmaz. Bunun nedeni final cevap token probability değerlerinin, reasoning sırasında cevabın daha önce yazılmasıyla yapay biçimde yükselmesini önlemektir.

In [5]:
def build_answer_messages(question: str) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen kısa ve doğrudan cevap veren bir soru-cevap sistemisin. "
                "Uzun açıklama veya gerekçe üretme."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıdaki olgusal soruyu Türkçe ve mümkün olduğunca kısa cevapla.\n\n"
                "Kurallar:\n"
                "- Yalnızca cevabı yaz.\n"
                "- Açıklama veya gerekçe ekleme.\n"
                "- Soruyu tekrar etme.\n"
                '- Cevabı bilmiyorsan yalnızca "Bilmiyorum" yaz.\n\n'
                f"Soru: {question}\n\n"
                "Cevap:"
            ),
        },
    ]


def build_confidence_messages(
    question: str,
    answer: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen verilen bir cevabın doğruluğunu değerlendiren bir sistemsin. "
                "İstenen çıktı formatına kesinlikle uy."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıda bir soru ve bu soruya verilmiş bir cevap bulunmaktadır.\n\n"
                f"Soru: {question}\n"
                f"Verilen cevap: {answer}\n\n"
                "Verilen cevabın tamamen doğru olma olasılığını "
                "0 ile 100 arasında değerlendir.\n\n"
                "Kurallar:\n"
                "- Yalnızca bir tam sayı yaz.\n"
                "- Yüzde işareti kullanma.\n"
                "- Açıklama veya gerekçe yazma.\n\n"
                "Güven:"
            ),
        },
    ]


def build_verification_messages(
    question: str,
    answer: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen olgusal cevapları dikkatle değerlendiren bir doğrulama sistemisin. "
                "Verilen cevap yanlışsa bunu açıkça belirt. "
                "Yalnızca izin verilen seçeneği üret."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıda bir soru ve bu soruya verilmiş bir cevap bulunmaktadır.\n\n"
                f"Soru: {question}\n"
                f"Verilen cevap: {answer}\n\n"
                "Bu cevap olgusal olarak tamamen doğru mudur?\n\n"
                "A) Doğru\n"
                "B) Yanlış\n"
                "C) Emin değilim\n\n"
                "Yalnızca A, B veya C yaz."
            ),
        },
    ]

# Aşama 6 — Deterministik cevap üretimi

In [6]:
@torch.inference_mode()
def generate_short_answer(
    question: str,
    max_new_tokens: int = ANSWER_MAX_NEW_TOKENS,
) -> dict:
    messages = build_answer_messages(question)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs.sequences[0, input_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "question": question,
        "prompt_text": prompt_text,
        "answer": answer,
        "generated_ids": generated_ids.detach().cpu(),
        "scores": outputs.scores,
        "input_token_count": int(input_length),
        "output_token_count": int(len(generated_ids)),
    }


@torch.inference_mode()
def generate_text_from_messages(
    messages: list[dict[str, str]],
    max_new_tokens: int,
) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0, input_length:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

# Aşama 7 — Token log-probability ve token entropy feature'ları

Bu aşama aşağıdaki sinyalleri üretir:

- `mean_logprob`
- `min_logprob`
- `max_logprob`
- `sequence_logprob`
- `geometric_mean_probability`
- `mean_token_entropy`
- `max_token_entropy`
- `first_token_entropy`
- Normalize edilmiş entropy sürümleri

`mean_logprob` ile `geometric_mean_probability` aynı bilginin monoton dönüşümüdür. Post-hoc modele ikisini birlikte vermiyoruz.

In [7]:
def extract_generated_token_statistics(
    result: dict,
    exclude_special_tokens: bool = True,
) -> dict:
    generated_ids = result["generated_ids"]
    scores = result["scores"]

    if len(generated_ids) != len(scores):
        raise ValueError(
            f"Üretilen token sayısı ({len(generated_ids)}) ile "
            f"score sayısı ({len(scores)}) eşleşmiyor."
        )

    all_token_records = []
    content_token_records = []
    vocab_size = None

    for position, (token_id_tensor, step_logits) in enumerate(
        zip(generated_ids, scores)
    ):
        token_id = int(token_id_tensor.item())
        is_special = token_id in tokenizer.all_special_ids

        logits = step_logits[0].float()
        vocab_size = int(logits.numel())

        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()

        token_logprob = float(log_probs[token_id].item())
        token_probability = float(math.exp(token_logprob))
        token_entropy = float(-(probs * log_probs).sum().item())
        normalized_entropy = float(
            token_entropy / math.log(vocab_size)
        )

        record = {
            "position": position,
            "token_id": token_id,
            "token": tokenizer.decode(
                [token_id],
                skip_special_tokens=False,
            ),
            "is_special": is_special,
            "logprob": token_logprob,
            "probability": token_probability,
            "entropy": token_entropy,
            "normalized_entropy": normalized_entropy,
        }

        all_token_records.append(record)

        if not (exclude_special_tokens and is_special):
            content_token_records.append(record)

    if not content_token_records:
        raise ValueError(
            "Özel tokenlar çıkarıldıktan sonra cevap tokenı kalmadı."
        )

    logprob_values = [
        record["logprob"]
        for record in content_token_records
    ]
    entropy_values = [
        record["entropy"]
        for record in content_token_records
    ]
    normalized_entropy_values = [
        record["normalized_entropy"]
        for record in content_token_records
    ]

    return {
        "all_tokens": all_token_records,
        "content_tokens": content_token_records,
        "content_token_count": len(content_token_records),
        "vocab_size": vocab_size,
        "mean_logprob": float(np.mean(logprob_values)),
        "min_logprob": float(np.min(logprob_values)),
        "max_logprob": float(np.max(logprob_values)),
        "sequence_logprob": float(np.sum(logprob_values)),
        "geometric_mean_probability": float(
            math.exp(np.mean(logprob_values))
        ),
        "mean_token_entropy": float(np.mean(entropy_values)),
        "max_token_entropy": float(np.max(entropy_values)),
        "first_token_entropy": float(entropy_values[0]),
        "mean_token_entropy_normalized": float(
            np.mean(normalized_entropy_values)
        ),
        "max_token_entropy_normalized": float(
            np.max(normalized_entropy_values)
        ),
        "first_token_entropy_normalized": float(
            normalized_entropy_values[0]
        ),
    }

# Aşama 8 — Self-reported confidence

In [8]:
def parse_confidence(raw_output: str) -> Optional[int]:
    text = raw_output.strip()

    if re.fullmatch(r"\d{1,3}", text):
        value = int(text)
        return value if 0 <= value <= 100 else None

    matches = re.findall(
        r"(?<!\d)(100|\d{1,2})(?!\d)",
        text,
    )

    if len(matches) != 1:
        return None

    value = int(matches[0])
    return value if 0 <= value <= 100 else None

# Aşama 9 — A/B/C self-verification ve restricted-choice olasılıkları

`verification_p_correct`, bütün vocabulary üzerinde kalibre edilmiş bir doğruluk olasılığı değildir. Yalnızca `A/B/C` hedef tokenları arasında normalize edilmiş bir sinyaldir.

In [9]:
def parse_verification_strict(
    raw_output: str,
) -> Optional[str]:
    text = raw_output.strip().upper()

    if text in {"A", "B", "C"}:
        return text

    match = re.search(r"\b([ABC])\b", text)
    return match.group(1) if match else None


def _single_token_variants(label: str) -> set[int]:
    variants = [
        label,
        f" {label}",
        f"\n{label}",
    ]
    token_ids = set()

    for variant in variants:
        encoded = tokenizer.encode(
            variant,
            add_special_tokens=False,
        )
        if len(encoded) == 1:
            token_ids.add(int(encoded[0]))

    if not token_ids:
        raise ValueError(
            f"{label} için tek tokenlı bir variant bulunamadı."
        )

    return token_ids


@torch.inference_mode()
def get_verification_probabilities(
    question: str,
    answer: str,
) -> dict:
    messages = build_verification_messages(
        question,
        answer,
    )

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model(**inputs)
    next_token_logits = outputs.logits[0, -1].float()
    next_token_probs = F.softmax(
        next_token_logits,
        dim=-1,
    )

    raw_probabilities = {}

    for label in ["A", "B", "C"]:
        token_ids = _single_token_variants(label)
        raw_probabilities[label] = float(
            sum(
                next_token_probs[token_id].item()
                for token_id in token_ids
            )
        )

    total = sum(raw_probabilities.values())

    if total <= 0:
        raise ValueError(
            "A/B/C restricted probability toplamı sıfır."
        )

    normalized = {
        label: probability / total
        for label, probability in raw_probabilities.items()
    }

    predicted_label = max(
        normalized,
        key=normalized.get,
    )

    p_a = normalized["A"]
    p_b = normalized["B"]
    p_c = normalized["C"]

    verification_margin = float(
        p_a - max(p_b, p_c)
    )

    verification_entropy = float(
        -sum(
            p * math.log(max(p, 1e-12))
            for p in [p_a, p_b, p_c]
        )
    )

    verification_entropy_normalized = float(
        verification_entropy / math.log(3)
    )

    return {
        "raw_probabilities": raw_probabilities,
        "normalized_probabilities": normalized,
        "predicted_label": predicted_label,
        "verification_margin": verification_margin,
        "verification_entropy": verification_entropy,
        "verification_entropy_normalized": (
            verification_entropy_normalized
        ),
    }

# Aşama 10 — Metin normalizasyonu, alias ön kontrolü ve cevap türü göstergeleri

In [10]:
UNKNOWN_PATTERNS = {
    "bilmiyorum",
    "emin değilim",
    "bilgim yok",
    "hatırlamıyorum",
    "cevabı bilmiyorum",
}

REFUSAL_PATTERNS = {
    "bu soruya cevap veremem",
    "bu soruyu yanıtlayamam",
    "yardımcı olamam",
    "cevap veremiyorum",
}


def normalize_text(text: str) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )
    text = text.casefold().strip()
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()
    return text


def is_unknown_response(text: str) -> bool:
    normalized = normalize_text(text)
    return (
        normalized in UNKNOWN_PATTERNS
        or any(
            normalized.startswith(pattern)
            for pattern in UNKNOWN_PATTERNS
        )
    )


def is_refusal_response(text: str) -> bool:
    normalized = normalize_text(text)
    return any(
        pattern in normalized
        for pattern in REFUSAL_PATTERNS
    )


def exact_alias_match(
    generated_answer: str,
    accepted_aliases: list[str],
) -> bool:
    generated = normalize_text(generated_answer)
    aliases = {
        normalize_text(alias)
        for alias in accepted_aliases
    }
    return generated in aliases


def contains_alias_match(
    generated_answer: str,
    accepted_aliases: list[str],
) -> bool:
    generated = normalize_text(generated_answer)

    return any(
        normalize_text(alias) in generated
        for alias in accepted_aliases
        if normalize_text(alias)
    )


def is_short_answer_format(answer: str) -> bool:
    stripped = answer.strip()

    if not stripped:
        return False

    nonempty_lines = [
        line
        for line in stripped.splitlines()
        if line.strip()
    ]

    return (
        len(nonempty_lines) == 1
        and len(stripped.split()) <= 12
    )

# Aşama 11 — Self-consistency ve davranışsal belirsizlik feature'ları

Self-consistency, sampled cevapların kaçının doğru olduğunu değil, en sık cevabın ne kadar tekrarlandığını ölçer.

Örneğin model beş kez yanlış biçimde `Bilmiyorum` derse:

- `self_consistency = 1.0`
- fakat cevap doğruluğu `0` olabilir.

Bu nedenle sampled cevapların referansa göre doğruluk oranı post-hoc feature olarak kullanılmaz.

In [11]:
@torch.inference_mode()
def generate_sampled_answer(
    question: str,
    seed: int,
    max_new_tokens: int = ANSWER_MAX_NEW_TOKENS,
    temperature: float = SAMPLE_TEMPERATURE,
    top_p: float = SAMPLE_TOP_P,
) -> str:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    messages = build_answer_messages(question)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0, input_length:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


def compute_self_consistency(
    question: str,
    deterministic_answer: str,
    seeds: list[int] = SELF_CONSISTENCY_SEEDS,
) -> dict:
    sampled_answers = [
        generate_sampled_answer(
            question=question,
            seed=seed,
        )
        for seed in seeds
    ]

    normalized_answers = [
        normalize_text(answer)
        for answer in sampled_answers
    ]

    counts = Counter(normalized_answers)

    majority_answer, majority_count = (
        counts.most_common(1)[0]
    )

    k = len(sampled_answers)
    cluster_probabilities = np.array(
        list(counts.values()),
        dtype=float,
    ) / k

    sample_entropy = float(
        -np.sum(
            cluster_probabilities
            * np.log(
                np.clip(
                    cluster_probabilities,
                    1e-12,
                    1.0,
                )
            )
        )
    )

    if len(counts) == 1:
        normalized_sample_entropy = 0.0
    else:
        normalized_sample_entropy = float(
            sample_entropy / math.log(k)
        )

    deterministic_normalized = normalize_text(
        deterministic_answer
    )

    unknown_count = sum(
        is_unknown_response(answer)
        for answer in sampled_answers
    )
    refusal_count = sum(
        is_refusal_response(answer)
        for answer in sampled_answers
    )

    return {
        "sampled_answers": sampled_answers,
        "normalized_answers": normalized_answers,
        "answer_counts": dict(counts),
        "majority_normalized_answer": majority_answer,
        "majority_count": int(majority_count),
        "self_consistency": float(
            majority_count / k
        ),
        "unique_answer_count": int(len(counts)),
        "unique_answer_ratio": float(
            len(counts) / k
        ),
        "sample_answer_entropy": sample_entropy,
        "sample_answer_entropy_normalized": (
            normalized_sample_entropy
        ),
        "deterministic_majority_agreement": int(
            deterministic_normalized
            == majority_answer
        ),
        "unknown_sample_rate": float(
            unknown_count / k
        ),
        "refusal_sample_rate": float(
            refusal_count / k
        ),
    }

# Aşama 12 — 60 soruluk Türkçe açık uçlu pilot seti

Sorular kısa, olgusal ve büyük ölçüde zamana dayanıklı olacak şekilde seçilmiştir. `difficulty` etiketi kullanılmaz. `category` ve `answer_type` yalnızca veri denetimi ve isteğe bağlı alt grup analizi içindir.

In [12]:
pilot_questions = [
    {
        "question_id": "Q01",
        "question": "Türkiye'nin başkenti neresidir?",
        "reference_answer": "Ankara",
        "accepted_aliases": [
            "Ankara"
        ],
        "answer_type": "LOCATION",
        "category": "TURKEY"
    },
    {
        "question_id": "Q02",
        "question": "Türkiye Cumhuriyeti hangi yıl ilan edildi?",
        "reference_answer": "1923",
        "accepted_aliases": [
            "1923",
            "29 Ekim 1923"
        ],
        "answer_type": "DATE",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q03",
        "question": "Dünya'nın doğal uydusunun adı nedir?",
        "reference_answer": "Ay",
        "accepted_aliases": [
            "Ay"
        ],
        "answer_type": "OBJECT",
        "category": "ASTRONOMY"
    },
    {
        "question_id": "Q04",
        "question": "Ay'a ilk ayak basan insan kimdir?",
        "reference_answer": "Neil Armstrong",
        "accepted_aliases": [
            "Neil Armstrong",
            "Neil Alden Armstrong"
        ],
        "answer_type": "PERSON",
        "category": "SPACE_HISTORY"
    },
    {
        "question_id": "Q05",
        "question": "Suyun kimyasal formülü nedir?",
        "reference_answer": "H2O",
        "accepted_aliases": [
            "H2O",
            "H₂O"
        ],
        "answer_type": "SCIENCE",
        "category": "CHEMISTRY"
    },
    {
        "question_id": "Q06",
        "question": "İstiklal Marşı'nın yazarı kimdir?",
        "reference_answer": "Mehmet Akif Ersoy",
        "accepted_aliases": [
            "Mehmet Akif Ersoy",
            "Mehmet Âkif Ersoy"
        ],
        "answer_type": "PERSON",
        "category": "TURKISH_LITERATURE"
    },
    {
        "question_id": "Q07",
        "question": "Bir üçgenin iç açılarının toplamı kaç derecedir?",
        "reference_answer": "180",
        "accepted_aliases": [
            "180",
            "180 derece"
        ],
        "answer_type": "NUMBER",
        "category": "MATHEMATICS"
    },
    {
        "question_id": "Q08",
        "question": "Güneş Sistemi'nin en büyük gezegeni hangisidir?",
        "reference_answer": "Jüpiter",
        "accepted_aliases": [
            "Jüpiter"
        ],
        "answer_type": "OBJECT",
        "category": "ASTRONOMY"
    },
    {
        "question_id": "Q09",
        "question": "Türkiye'nin en kalabalık şehri hangisidir?",
        "reference_answer": "İstanbul",
        "accepted_aliases": [
            "İstanbul"
        ],
        "answer_type": "LOCATION",
        "category": "TURKEY_GEOGRAPHY"
    },
    {
        "question_id": "Q10",
        "question": "DNA'nın Türkçe açılımı nedir?",
        "reference_answer": "Deoksiribonükleik asit",
        "accepted_aliases": [
            "Deoksiribonükleik asit",
            "Deoksiribonükleik asit molekülü"
        ],
        "answer_type": "CONCEPT",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q11",
        "question": "Osmanlı Devleti'nin kurucusu kimdir?",
        "reference_answer": "Osman Gazi",
        "accepted_aliases": [
            "Osman Gazi",
            "I. Osman",
            "Osman Bey"
        ],
        "answer_type": "PERSON",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q12",
        "question": "Lozan Barış Antlaşması hangi yıl imzalandı?",
        "reference_answer": "1923",
        "accepted_aliases": [
            "1923",
            "24 Temmuz 1923"
        ],
        "answer_type": "DATE",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q13",
        "question": "Periyodik tabloda atom numarası 8 olan element hangisidir?",
        "reference_answer": "Oksijen",
        "accepted_aliases": [
            "Oksijen",
            "O"
        ],
        "answer_type": "SCIENCE",
        "category": "CHEMISTRY"
    },
    {
        "question_id": "Q14",
        "question": "Suç ve Ceza romanının yazarı kimdir?",
        "reference_answer": "Fyodor Dostoyevski",
        "accepted_aliases": [
            "Fyodor Dostoyevski",
            "Dostoyevski",
            "Fyodor Mihayloviç Dostoyevski"
        ],
        "answer_type": "PERSON",
        "category": "WORLD_LITERATURE"
    },
    {
        "question_id": "Q15",
        "question": "Işık boşlukta yaklaşık saniyede kaç kilometre yol alır?",
        "reference_answer": "300000",
        "accepted_aliases": [
            "300000",
            "300.000",
            "299792",
            "299.792",
            "300000 km",
            "300.000 km",
            "300 bin kilometre"
        ],
        "answer_type": "NUMBER",
        "category": "PHYSICS"
    },
    {
        "question_id": "Q16",
        "question": "Magna Carta hangi ülkede imzalanmıştır?",
        "reference_answer": "İngiltere",
        "accepted_aliases": [
            "İngiltere",
            "England"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_HISTORY"
    },
    {
        "question_id": "Q17",
        "question": "TCP kısaltmasının açılımı nedir?",
        "reference_answer": "Transmission Control Protocol",
        "accepted_aliases": [
            "Transmission Control Protocol",
            "İletim Kontrol Protokolü"
        ],
        "answer_type": "CONCEPT",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q18",
        "question": "Python programlama dilinin yaratıcısı kimdir?",
        "reference_answer": "Guido van Rossum",
        "accepted_aliases": [
            "Guido van Rossum",
            "Guido Van Rossum"
        ],
        "answer_type": "PERSON",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q19",
        "question": "Fotosentez sırasında bitkiler atmosferden hangi gazı alır?",
        "reference_answer": "Karbondioksit",
        "accepted_aliases": [
            "Karbondioksit",
            "CO2",
            "CO₂"
        ],
        "answer_type": "SCIENCE",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q20",
        "question": "İnsan vücudundaki en büyük organ hangisidir?",
        "reference_answer": "Deri",
        "accepted_aliases": [
            "Deri",
            "Cilt"
        ],
        "answer_type": "SCIENCE",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q21",
        "question": "Nutuk adlı eserin yazarı kimdir?",
        "reference_answer": "Mustafa Kemal Atatürk",
        "accepted_aliases": [
            "Mustafa Kemal Atatürk",
            "Atatürk",
            "Gazi Mustafa Kemal Atatürk"
        ],
        "answer_type": "PERSON",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q22",
        "question": "Türkiye'nin resmî para birimi nedir?",
        "reference_answer": "Türk lirası",
        "accepted_aliases": [
            "Türk lirası",
            "TL",
            "TRY"
        ],
        "answer_type": "CONCEPT",
        "category": "TURKEY"
    },
    {
        "question_id": "Q23",
        "question": "İstanbul Boğazı hangi iki denizi birbirine bağlar?",
        "reference_answer": "Karadeniz ve Marmara Denizi",
        "accepted_aliases": [
            "Karadeniz ve Marmara Denizi",
            "Marmara Denizi ve Karadeniz",
            "Karadeniz ile Marmara Denizi"
        ],
        "answer_type": "LOCATION",
        "category": "TURKEY_GEOGRAPHY"
    },
    {
        "question_id": "Q24",
        "question": "Everest Dağı hangi kıtadadır?",
        "reference_answer": "Asya",
        "accepted_aliases": [
            "Asya",
            "Asya kıtası"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q25",
        "question": "Nil Nehri hangi denize dökülür?",
        "reference_answer": "Akdeniz",
        "accepted_aliases": [
            "Akdeniz",
            "Akdeniz'e"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q26",
        "question": "Japonya'nın başkenti neresidir?",
        "reference_answer": "Tokyo",
        "accepted_aliases": [
            "Tokyo"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q27",
        "question": "Kanada'nın başkenti neresidir?",
        "reference_answer": "Ottawa",
        "accepted_aliases": [
            "Ottawa"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q28",
        "question": "Gize Piramitleri hangi ülkededir?",
        "reference_answer": "Mısır",
        "accepted_aliases": [
            "Mısır",
            "Mısır Arap Cumhuriyeti",
            "Egypt"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q29",
        "question": "Dünya'nın en büyük okyanusu hangisidir?",
        "reference_answer": "Pasifik Okyanusu",
        "accepted_aliases": [
            "Pasifik Okyanusu",
            "Büyük Okyanus",
            "Pasifik"
        ],
        "answer_type": "LOCATION",
        "category": "WORLD_GEOGRAPHY"
    },
    {
        "question_id": "Q30",
        "question": "Kızıl Gezegen olarak bilinen gezegen hangisidir?",
        "reference_answer": "Mars",
        "accepted_aliases": [
            "Mars"
        ],
        "answer_type": "OBJECT",
        "category": "ASTRONOMY"
    },
    {
        "question_id": "Q31",
        "question": "Au kimyasal sembolü hangi elementi temsil eder?",
        "reference_answer": "Altın",
        "accepted_aliases": [
            "Altın",
            "Gold"
        ],
        "answer_type": "SCIENCE",
        "category": "CHEMISTRY"
    },
    {
        "question_id": "Q32",
        "question": "Na kimyasal sembolü hangi elementi temsil eder?",
        "reference_answer": "Sodyum",
        "accepted_aliases": [
            "Sodyum",
            "Sodium"
        ],
        "answer_type": "SCIENCE",
        "category": "CHEMISTRY"
    },
    {
        "question_id": "Q33",
        "question": "Su deniz seviyesinde kaç santigrat derecede donar?",
        "reference_answer": "0",
        "accepted_aliases": [
            "0",
            "0 derece",
            "0 °C",
            "0 santigrat derece"
        ],
        "answer_type": "NUMBER",
        "category": "PHYSICS"
    },
    {
        "question_id": "Q34",
        "question": "Su deniz seviyesinde kaç santigrat derecede kaynar?",
        "reference_answer": "100",
        "accepted_aliases": [
            "100",
            "100 derece",
            "100 °C",
            "100 santigrat derece"
        ],
        "answer_type": "NUMBER",
        "category": "PHYSICS"
    },
    {
        "question_id": "Q35",
        "question": "Cisimleri Dünya'nın merkezine doğru çeken kuvvetin adı nedir?",
        "reference_answer": "Yerçekimi",
        "accepted_aliases": [
            "Yerçekimi",
            "Yer çekimi",
            "Kütle çekimi",
            "Kütleçekim"
        ],
        "answer_type": "CONCEPT",
        "category": "PHYSICS"
    },
    {
        "question_id": "Q36",
        "question": "İnsan vücudunda kanı pompalayan organ hangisidir?",
        "reference_answer": "Kalp",
        "accepted_aliases": [
            "Kalp"
        ],
        "answer_type": "SCIENCE",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q37",
        "question": "Kanda oksijen taşımakla görevli hücrelerin adı nedir?",
        "reference_answer": "Alyuvarlar",
        "accepted_aliases": [
            "Alyuvarlar",
            "Eritrositler",
            "Kırmızı kan hücreleri",
            "Alyuvar"
        ],
        "answer_type": "SCIENCE",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q38",
        "question": "İnsan vücudundaki en uzun kemik hangisidir?",
        "reference_answer": "Femur",
        "accepted_aliases": [
            "Femur",
            "Uyluk kemiği",
            "Femur kemiği"
        ],
        "answer_type": "SCIENCE",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q39",
        "question": "Kalıtımın temel birimine ne ad verilir?",
        "reference_answer": "Gen",
        "accepted_aliases": [
            "Gen"
        ],
        "answer_type": "CONCEPT",
        "category": "BIOLOGY"
    },
    {
        "question_id": "Q40",
        "question": "Bir sıvının gaz hâline geçmesine ne ad verilir?",
        "reference_answer": "Buharlaşma",
        "accepted_aliases": [
            "Buharlaşma"
        ],
        "answer_type": "CONCEPT",
        "category": "PHYSICS"
    },
    {
        "question_id": "Q41",
        "question": "Hamlet adlı eserin yazarı kimdir?",
        "reference_answer": "William Shakespeare",
        "accepted_aliases": [
            "William Shakespeare",
            "Shakespeare"
        ],
        "answer_type": "PERSON",
        "category": "WORLD_LITERATURE"
    },
    {
        "question_id": "Q42",
        "question": "Don Kişot romanının yazarı kimdir?",
        "reference_answer": "Miguel de Cervantes",
        "accepted_aliases": [
            "Miguel de Cervantes",
            "Cervantes",
            "Miguel Cervantes"
        ],
        "answer_type": "PERSON",
        "category": "WORLD_LITERATURE"
    },
    {
        "question_id": "Q43",
        "question": "Mona Lisa tablosunun ressamı kimdir?",
        "reference_answer": "Leonardo da Vinci",
        "accepted_aliases": [
            "Leonardo da Vinci",
            "Leonardo Da Vinci",
            "Leonardo"
        ],
        "answer_type": "PERSON",
        "category": "ART"
    },
    {
        "question_id": "Q44",
        "question": "Guernica tablosunun ressamı kimdir?",
        "reference_answer": "Pablo Picasso",
        "accepted_aliases": [
            "Pablo Picasso",
            "Picasso"
        ],
        "answer_type": "PERSON",
        "category": "ART"
    },
    {
        "question_id": "Q45",
        "question": "Für Elise adlı eserin bestecisi kimdir?",
        "reference_answer": "Ludwig van Beethoven",
        "accepted_aliases": [
            "Ludwig van Beethoven",
            "Beethoven",
            "Ludwig van Beethoven"
        ],
        "answer_type": "PERSON",
        "category": "MUSIC"
    },
    {
        "question_id": "Q46",
        "question": "İnce Memed romanının yazarı kimdir?",
        "reference_answer": "Yaşar Kemal",
        "accepted_aliases": [
            "Yaşar Kemal"
        ],
        "answer_type": "PERSON",
        "category": "TURKISH_LITERATURE"
    },
    {
        "question_id": "Q47",
        "question": "Kürk Mantolu Madonna romanının yazarı kimdir?",
        "reference_answer": "Sabahattin Ali",
        "accepted_aliases": [
            "Sabahattin Ali"
        ],
        "answer_type": "PERSON",
        "category": "TURKISH_LITERATURE"
    },
    {
        "question_id": "Q48",
        "question": "Divanü Lügati't-Türk adlı eserin yazarı kimdir?",
        "reference_answer": "Kaşgarlı Mahmud",
        "accepted_aliases": [
            "Kaşgarlı Mahmud",
            "Mahmud el-Kaşgarî",
            "Mahmud Kaşgari"
        ],
        "answer_type": "PERSON",
        "category": "TURKISH_LITERATURE"
    },
    {
        "question_id": "Q49",
        "question": "Türkiye Cumhuriyeti'nin ilk cumhurbaşkanı kimdir?",
        "reference_answer": "Mustafa Kemal Atatürk",
        "accepted_aliases": [
            "Mustafa Kemal Atatürk",
            "Atatürk",
            "Gazi Mustafa Kemal Atatürk"
        ],
        "answer_type": "PERSON",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q50",
        "question": "Malazgirt Savaşı hangi yıl yapılmıştır?",
        "reference_answer": "1071",
        "accepted_aliases": [
            "1071"
        ],
        "answer_type": "DATE",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q51",
        "question": "İstanbul'un fethi hangi yıl gerçekleşmiştir?",
        "reference_answer": "1453",
        "accepted_aliases": [
            "1453"
        ],
        "answer_type": "DATE",
        "category": "TURKEY_HISTORY"
    },
    {
        "question_id": "Q52",
        "question": "Fransız İhtilali hangi yıl başlamıştır?",
        "reference_answer": "1789",
        "accepted_aliases": [
            "1789"
        ],
        "answer_type": "DATE",
        "category": "WORLD_HISTORY"
    },
    {
        "question_id": "Q53",
        "question": "Birleşmiş Milletler hangi yıl kurulmuştur?",
        "reference_answer": "1945",
        "accepted_aliases": [
            "1945",
            "24 Ekim 1945"
        ],
        "answer_type": "DATE",
        "category": "WORLD_HISTORY"
    },
    {
        "question_id": "Q54",
        "question": "Roma rakamlarında X hangi sayıyı ifade eder?",
        "reference_answer": "10",
        "accepted_aliases": [
            "10",
            "On"
        ],
        "answer_type": "NUMBER",
        "category": "MATHEMATICS"
    },
    {
        "question_id": "Q55",
        "question": "144 sayısının karekökü kaçtır?",
        "reference_answer": "12",
        "accepted_aliases": [
            "12",
            "On iki"
        ],
        "answer_type": "NUMBER",
        "category": "MATHEMATICS"
    },
    {
        "question_id": "Q56",
        "question": "Onluk sistemdeki 2 sayısının ikilik sistemde gösterimi nedir?",
        "reference_answer": "10",
        "accepted_aliases": [
            "10",
            "10₂",
            "0b10"
        ],
        "answer_type": "NUMBER",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q57",
        "question": "HTML kısaltmasının açılımı nedir?",
        "reference_answer": "HyperText Markup Language",
        "accepted_aliases": [
            "HyperText Markup Language",
            "Hypertext Markup Language",
            "Köprü Metni Biçimlendirme Dili"
        ],
        "answer_type": "CONCEPT",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q58",
        "question": "HTTP kısaltmasının açılımı nedir?",
        "reference_answer": "HyperText Transfer Protocol",
        "accepted_aliases": [
            "HyperText Transfer Protocol",
            "Hypertext Transfer Protocol",
            "Köprü Metni Aktarım Protokolü"
        ],
        "answer_type": "CONCEPT",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q59",
        "question": "Linux çekirdeğinin ilk geliştiricisi kimdir?",
        "reference_answer": "Linus Torvalds",
        "accepted_aliases": [
            "Linus Torvalds"
        ],
        "answer_type": "PERSON",
        "category": "COMPUTER_SCIENCE"
    },
    {
        "question_id": "Q60",
        "question": "İlişkisel veritabanlarında veri sorgulamak için yaygın kullanılan dilin kısaltması nedir?",
        "reference_answer": "SQL",
        "accepted_aliases": [
            "SQL",
            "Structured Query Language",
            "Yapılandırılmış Sorgu Dili"
        ],
        "answer_type": "CONCEPT",
        "category": "COMPUTER_SCIENCE"
    }
]

assert len(pilot_questions) == 60
assert len({item["question_id"] for item in pilot_questions}) == 60

questions_df = pd.DataFrame(pilot_questions)

print("Toplam soru:", len(questions_df))
print("\nAnswer type dağılımı:")
print(questions_df["answer_type"].value_counts())
print("\nKategori dağılımı:")
print(questions_df["category"].value_counts())

Toplam soru: 60

Answer type dağılımı:
answer_type
PERSON      16
LOCATION    10
SCIENCE      9
CONCEPT      9
NUMBER       7
DATE         6
OBJECT       3
Name: count, dtype: int64

Kategori dağılımı:
category
TURKEY_HISTORY        7
BIOLOGY               7
COMPUTER_SCIENCE      7
WORLD_GEOGRAPHY       6
PHYSICS               5
CHEMISTRY             4
TURKISH_LITERATURE    4
ASTRONOMY             3
WORLD_HISTORY         3
WORLD_LITERATURE      3
MATHEMATICS           3
TURKEY                2
ART                   2
TURKEY_GEOGRAPHY      2
SPACE_HISTORY         1
MUSIC                 1
Name: count, dtype: int64


# Aşama 13 — Tüm feature'ları çıkaran birleşik soru pipeline'ı

In [13]:
def run_single_pilot_question(
    item: dict,
) -> dict:
    question = item["question"]
    start_time = time.time()

    # 1. Deterministik ana cevap
    answer_result = generate_short_answer(
        question=question,
    )
    generated_answer = answer_result["answer"]

    # 2. Token log-probability + entropy
    token_stats = extract_generated_token_statistics(
        answer_result,
        exclude_special_tokens=True,
    )

    # 3. Sayısal self-confidence
    confidence_raw = generate_text_from_messages(
        build_confidence_messages(
            question=question,
            answer=generated_answer,
        ),
        max_new_tokens=CONFIDENCE_MAX_NEW_TOKENS,
    )
    confidence_value = parse_confidence(
        confidence_raw
    )
    confidence_scaled = (
        confidence_value / 100
        if confidence_value is not None
        else np.nan
    )

    # 4. Kategorik self-verification
    verification_raw = generate_text_from_messages(
        build_verification_messages(
            question=question,
            answer=generated_answer,
        ),
        max_new_tokens=VERIFICATION_MAX_NEW_TOKENS,
    )
    verification_label = (
        parse_verification_strict(
            verification_raw
        )
    )

    # 5. A/B/C restricted-choice probabilities
    verification = (
        get_verification_probabilities(
            question=question,
            answer=generated_answer,
        )
    )
    verification_p = (
        verification[
            "normalized_probabilities"
        ]
    )

    # 6. Self-consistency
    if RUN_SELF_CONSISTENCY:
        consistency = compute_self_consistency(
            question=question,
            deterministic_answer=generated_answer,
        )
    else:
        consistency = {
            "sampled_answers": [],
            "normalized_answers": [],
            "answer_counts": {},
            "majority_normalized_answer": "",
            "majority_count": np.nan,
            "self_consistency": np.nan,
            "unique_answer_count": np.nan,
            "unique_answer_ratio": np.nan,
            "sample_answer_entropy": np.nan,
            "sample_answer_entropy_normalized": np.nan,
            "deterministic_majority_agreement": np.nan,
            "unknown_sample_rate": np.nan,
            "refusal_sample_rate": np.nan,
        }

    # 7. Ground-truth tabanlı otomatik ön kontroller
    exact_match = exact_alias_match(
        generated_answer,
        item["accepted_aliases"],
    )
    contains_match = contains_alias_match(
        generated_answer,
        item["accepted_aliases"],
    )

    elapsed_seconds = time.time() - start_time

    return {
        # Kimlik ve metadata
        "question_id": item["question_id"],
        "question": question,
        "reference_answer": (
            item["reference_answer"]
        ),
        "accepted_aliases": " | ".join(
            item["accepted_aliases"]
        ),
        "answer_type": item["answer_type"],
        "category": item["category"],
        "model_id": MODEL_ID,
        "quantization": QUANTIZATION_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "answer_prompt_version": "answer_v2",
        "confidence_prompt_version": "confidence_v1",
        "verification_prompt_version": "verification_v1",

        # Ana cevap
        "generated_answer": generated_answer,
        "normalized_answer": normalize_text(
            generated_answer
        ),
        "answer_format_compliant": int(
            is_short_answer_format(
                generated_answer
            )
        ),
        "unknown_response": int(
            is_unknown_response(
                generated_answer
            )
        ),
        "refusal_indicator": int(
            is_refusal_response(
                generated_answer
            )
        ),

        # Ground-truth tabanlı ön kontrol:
        # post-hoc modele feature olarak verilmez.
        "automatic_exact_alias_match": int(
            exact_match
        ),
        "automatic_contains_alias_match": int(
            contains_match
        ),

        # Manuel etiket alanları
        "manual_is_correct": np.nan,
        "manual_error_type": "",
        "manual_notes": "",

        # Uzunluk
        "input_token_count": (
            answer_result[
                "input_token_count"
            ]
        ),
        "output_token_count_all": (
            answer_result[
                "output_token_count"
            ]
        ),
        "output_content_token_count": (
            token_stats[
                "content_token_count"
            ]
        ),

        # Token log-probability
        "mean_logprob": (
            token_stats["mean_logprob"]
        ),
        "min_logprob": (
            token_stats["min_logprob"]
        ),
        "max_logprob": (
            token_stats["max_logprob"]
        ),
        "sequence_logprob": (
            token_stats[
                "sequence_logprob"
            ]
        ),
        "geometric_mean_probability": (
            token_stats[
                "geometric_mean_probability"
            ]
        ),

        # Token entropy
        "mean_token_entropy": (
            token_stats[
                "mean_token_entropy"
            ]
        ),
        "max_token_entropy": (
            token_stats[
                "max_token_entropy"
            ]
        ),
        "first_token_entropy": (
            token_stats[
                "first_token_entropy"
            ]
        ),
        "mean_token_entropy_normalized": (
            token_stats[
                "mean_token_entropy_normalized"
            ]
        ),
        "max_token_entropy_normalized": (
            token_stats[
                "max_token_entropy_normalized"
            ]
        ),
        "first_token_entropy_normalized": (
            token_stats[
                "first_token_entropy_normalized"
            ]
        ),

        # Self-reported confidence
        "self_confidence_raw": (
            confidence_raw
        ),
        "self_confidence_value": (
            confidence_value
        ),
        "self_confidence_scaled": (
            confidence_scaled
        ),
        "confidence_parse_success": int(
            confidence_value is not None
        ),
        "confidence_format_compliant": int(
            bool(
                re.fullmatch(
                    r"\d{1,3}",
                    confidence_raw.strip(),
                )
            )
        ),

        # Self-verification
        "verification_raw": verification_raw,
        "verification_label": (
            verification_label
        ),
        "verification_parse_success": int(
            verification_label is not None
        ),
        "verification_p_correct": (
            verification_p["A"]
        ),
        "verification_p_incorrect": (
            verification_p["B"]
        ),
        "verification_p_uncertain": (
            verification_p["C"]
        ),
        "verification_probability_prediction": (
            verification[
                "predicted_label"
            ]
        ),
        "verification_margin": (
            verification[
                "verification_margin"
            ]
        ),
        "verification_entropy": (
            verification[
                "verification_entropy"
            ]
        ),
        "verification_entropy_normalized": (
            verification[
                "verification_entropy_normalized"
            ]
        ),

        # Self-consistency
        "sampled_answers_json": json.dumps(
            consistency["sampled_answers"],
            ensure_ascii=False,
        ),
        "sample_answer_counts_json": json.dumps(
            consistency["answer_counts"],
            ensure_ascii=False,
        ),
        "majority_normalized_answer": (
            consistency[
                "majority_normalized_answer"
            ]
        ),
        "majority_count": (
            consistency["majority_count"]
        ),
        "self_consistency": (
            consistency[
                "self_consistency"
            ]
        ),
        "unique_answer_count": (
            consistency[
                "unique_answer_count"
            ]
        ),
        "unique_answer_ratio": (
            consistency[
                "unique_answer_ratio"
            ]
        ),
        "sample_answer_entropy": (
            consistency[
                "sample_answer_entropy"
            ]
        ),
        "sample_answer_entropy_normalized": (
            consistency[
                "sample_answer_entropy_normalized"
            ]
        ),
        "deterministic_majority_agreement": (
            consistency[
                "deterministic_majority_agreement"
            ]
        ),
        "unknown_sample_rate": (
            consistency[
                "unknown_sample_rate"
            ]
        ),
        "refusal_sample_rate": (
            consistency[
                "refusal_sample_rate"
            ]
        ),

        # Teknik
        "elapsed_seconds": (
            elapsed_seconds
        ),
    }

# Aşama 14 — Tek soru sanity check

Önce yalnızca Q01 çalıştırılır. Çıktı doğru görünmeden 60 soruluk döngüye geçmeyin.

In [14]:
single_test = run_single_pilot_question(
    pilot_questions[0]
)

important_fields = [
    "question_id",
    "generated_answer",
    "mean_logprob",
    "min_logprob",
    "mean_token_entropy_normalized",
    "self_confidence_scaled",
    "verification_p_correct",
    "verification_margin",
    "verification_entropy_normalized",
    "self_consistency",
    "unique_answer_count",
    "sample_answer_entropy_normalized",
    "deterministic_majority_agreement",
    "unknown_response",
    "unknown_sample_rate",
    "elapsed_seconds",
]

{
    field: single_test[field]
    for field in important_fields
}

{'question_id': 'Q01',
 'generated_answer': 'Ankara.',
 'mean_logprob': -0.014461927694355836,
 'min_logprob': -0.05516854673624039,
 'mean_token_entropy_normalized': 0.005395396597602179,
 'self_confidence_scaled': 1.0,
 'verification_p_correct': 0.9999999982807275,
 'verification_margin': 0.9999999973942365,
 'verification_entropy_normalized': 3.423173648342652e-08,
 'self_consistency': 1.0,
 'unique_answer_count': 1,
 'sample_answer_entropy_normalized': 0.0,
 'deterministic_majority_agreement': 1,
 'unknown_response': 0,
 'unknown_sample_rate': 0.0,
 'elapsed_seconds': 5.236069917678833}

# Aşama 15 — 60 soruluk batch çalıştırma ve checkpoint

- Her başarılı soru sonrasında CSV yeniden yazılır.
- Colab bağlantısı kesilirse aynı hücre yeniden çalıştırılabilir.
- `question_id` daha önce tamamlandıysa soru atlanır.
- Feature şeması değiştirildiği için v2 checkpoint dosyası kullanılır.

In [15]:
results = []

if (
    RESUME_FROM_CHECKPOINT
    and RAW_RESULTS_PATH.exists()
):
    existing_df = pd.read_csv(
        RAW_RESULTS_PATH
    )
    results = existing_df.to_dict(
        "records"
    )
    print(
        "Checkpoint yüklendi:",
        len(results),
        "soru",
    )

completed_ids = {
    row["question_id"]
    for row in results
}

for index, item in enumerate(
    pilot_questions,
    start=1,
):
    question_id = item["question_id"]

    if question_id in completed_ids:
        print(
            f"[{index}/60] {question_id} "
            "checkpoint'te mevcut, atlandı."
        )
        continue

    print(
        f"[{index}/60] {question_id}: "
        f"{item['question']}"
    )

    try:
        result = run_single_pilot_question(
            item
        )
        results.append(result)
        completed_ids.add(question_id)

        pd.DataFrame(results).to_csv(
            RAW_RESULTS_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        print(
            "  Cevap:",
            repr(
                result[
                    "generated_answer"
                ]
            ),
        )
        print(
            "  Mean LP:",
            round(
                result["mean_logprob"],
                4,
            ),
        )
        print(
            "  Entropy(norm):",
            round(
                result[
                    "mean_token_entropy_normalized"
                ],
                4,
            ),
        )
        print(
            "  Self-conf:",
            result[
                "self_confidence_scaled"
            ],
        )
        print(
            "  Verify P(correct):",
            round(
                result[
                    "verification_p_correct"
                ],
                4,
            ),
        )
        print(
            "  Self-consistency:",
            result[
                "self_consistency"
            ],
        )
        print(
            "  Süre:",
            round(
                result[
                    "elapsed_seconds"
                ],
                2,
            ),
            "sn\n",
        )

    except Exception as error:
        if results:
            pd.DataFrame(results).to_csv(
                RAW_RESULTS_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        print(
            f"HATA — {question_id}: "
            f"{type(error).__name__}: "
            f"{error}"
        )
        raise

print(
    "Tamamlandı:",
    len(results),
    "kayıt",
)
print("CSV:", RAW_RESULTS_PATH)

[1/60] Q01: Türkiye'nin başkenti neresidir?
  Cevap: 'Ankara.'
  Mean LP: -0.0145
  Entropy(norm): 0.0054
  Self-conf: 1.0
  Verify P(correct): 1.0
  Self-consistency: 1.0
  Süre: 3.61 sn

[2/60] Q02: Türkiye Cumhuriyeti hangi yıl ilan edildi?
  Cevap: '1923.'
  Mean LP: -0.1223
  Entropy(norm): 0.0204
  Self-conf: 0.9
  Verify P(correct): 1.0
  Self-consistency: 0.6
  Süre: 4.7 sn

[3/60] Q03: Dünya'nın doğal uydusunun adı nedir?
  Cevap: 'Bilmiyorum.'
  Mean LP: -0.0004
  Entropy(norm): 0.0004
  Self-conf: 0.5
  Verify P(correct): 0.0
  Self-consistency: 1.0
  Süre: 4.67 sn

[4/60] Q04: Ay'a ilk ayak basan insan kimdir?
  Cevap: 'Neil Armstrong.'
  Mean LP: -0.0357
  Entropy(norm): 0.0104
  Self-conf: 0.85
  Verify P(correct): 1.0
  Self-consistency: 1.0
  Süre: 6.89 sn

[5/60] Q05: Suyun kimyasal formülü nedir?
  Cevap: 'H₂O'
  Mean LP: -0.0012
  Entropy(norm): 0.0007
  Self-conf: 0.95
  Verify P(correct): 1.0
  Self-consistency: 1.0
  Süre: 3.19 sn

[6/60] Q06: İstiklal Marşı'nın y

# Aşama 16 — Ham sonuçların kontrolü

In [16]:
raw_df = pd.read_csv(
    RAW_RESULTS_PATH
)

display_columns = [
    "question_id",
    "question",
    "reference_answer",
    "generated_answer",
    "automatic_exact_alias_match",
    "automatic_contains_alias_match",
    "unknown_response",
    "mean_logprob",
    "mean_token_entropy_normalized",
    "self_confidence_scaled",
    "verification_p_correct",
    "verification_margin",
    "self_consistency",
    "unique_answer_count",
    "sample_answer_entropy_normalized",
    "unknown_sample_rate",
]

display(
    raw_df[display_columns]
)

print("Toplam kayıt:", len(raw_df))
print(
    "Format uyumu:",
    raw_df[
        "answer_format_compliant"
    ].mean(),
)
print(
    "Confidence parse:",
    raw_df[
        "confidence_parse_success"
    ].mean(),
)
print(
    "Verification parse:",
    raw_df[
        "verification_parse_success"
    ].mean(),
)
print(
    "Exact alias ön eşleşme:",
    raw_df[
        "automatic_exact_alias_match"
    ].mean(),
)
print(
    "Contains alias ön eşleşme:",
    raw_df[
        "automatic_contains_alias_match"
    ].mean(),
)
print(
    "Unknown cevap sayısı:",
    raw_df[
        "unknown_response"
    ].sum(),
)

,question_id,question,reference_answer,generated_answer,automatic_exact_alias_match,automatic_contains_alias_match,unknown_response,mean_logprob,mean_token_entropy_normalized,self_confidence_scaled,verification_p_correct,verification_margin,self_consistency,unique_answer_count,sample_answer_entropy_normalized,unknown_sample_rate
0,Q01,Türkiye'nin başkenti neresidir?,Ankara,Ankara.,1,1,0,-0.014462,0.005395,1.00,1.000000e+00,1.000000,1.0,1,0.000000,0.0
1,Q02,Türkiye Cumhuriyeti hangi yıl ilan edildi?,1923,1923.,1,1,0,-0.122272,0.020432,0.90,9.999993e-01,0.999999,0.6,2,0.418166,0.0
2,Q03,Dünya'nın doğal uydusunun adı nedir?,Ay,Bilmiyorum.,0,0,1,-0.000428,0.000362,0.50,6.796959e-10,-0.999999,1.0,1,0.000000,1.0
3,Q04,Ay'a ilk ayak basan insan kimdir?,Neil Armstrong,Neil Armstrong.,1,1,0,-0.035664,0.010418,0.85,1.000000e+00,1.000000,1.0,1,0.000000,0.0
4,Q05,Suyun kimyasal formülü nedir?,H2O,H₂O,1,1,0,-0.001190,0.000666,0.95,1.000000e+00,1.000000,1.0,1,0.000000,0.0
5,Q06,İstiklal Marşı'nın yazarı kimdir?,Mehmet Akif Ersoy,Mustafa Kemal Atatürk.,0,0,0,-0.161436,0.035349,0.85,2.873677e-07,-0.999996,0.6,3,0.590436,0.0
6,Q07,Bir üçgenin iç açılarının toplamı kaç derecedir?,180,180 derece.,1,1,0,-0.007366,0.002811,0.90,1.000000e+00,1.000000,1.0,1,0.000000,0.0
7,Q08,Güneş Sistemi'nin en büyük gezegeni hangisidir?,Jüpiter,Jüpiter,1,1,0,-0.093175,0.020471,0.90,1.000000e+00,1.000000,1.0,1,0.000000,0.0
8,Q09,Türkiye'nin en kalabalık şehri hangisidir?,İstanbul,İstanbul,1,1,0,-0.215118,0.031548,0.70,9.998785e-01,0.999763,0.8,2,0.310918,0.2
9,Q10,DNA'nın Türkçe açılımı nedir?,Deoksiribonükleik asit,DNA: Deoksiribonükleik Asit.,0,1,0,-0.058498,0.013944,0.95,1.000000e+00,1.000000,0.6,3,0.590436,0.0


Toplam kayıt: 60
Format uyumu: 0.9833333333333333
Confidence parse: 1.0
Verification parse: 1.0
Exact alias ön eşleşme: 0.5333333333333333
Contains alias ön eşleşme: 0.65
Unknown cevap sayısı: 12


# Aşama 17 — Manuel etiketleme şablonu

Nihai hedef değişken `manual_is_correct` olmalıdır.

Önerilen hata türleri:

- `NONE`
- `FACTUAL_ERROR`
- `UNKNOWN_RESPONSE`
- `REFUSAL`
- `NON_RESPONSIVE`
- `NONSENSICAL`
- `FORMAT_FAILURE`
- `AMBIGUOUS`

`automatic_exact_alias_match` ve `automatic_contains_alias_match` yalnızca ön kontrol içindir. Nihai etiket yerine doğrudan kullanılmamalıdır.

In [17]:
annotation_columns = [
    "question_id",
    "question",
    "reference_answer",
    "accepted_aliases",
    "generated_answer",
    "automatic_exact_alias_match",
    "automatic_contains_alias_match",
    "manual_is_correct",
    "manual_error_type",
    "manual_notes",
]

annotation_df = raw_df[
    annotation_columns
].copy()

annotation_df.to_csv(
    ANNOTATION_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(annotation_df)

print(
    "Etiketleme şablonu:",
    ANNOTATION_TEMPLATE_PATH,
)
print(
    "CSV'yi indirip manual_is_correct, "
    "manual_error_type ve manual_notes "
    "alanlarını doldurun."
)

,question_id,question,reference_answer,accepted_aliases,generated_answer,automatic_exact_alias_match,automatic_contains_alias_match,manual_is_correct,manual_error_type,manual_notes
0,Q01,Türkiye'nin başkenti neresidir?,Ankara,Ankara,Ankara.,1,1,NaN,NaN,NaN
1,Q02,Türkiye Cumhuriyeti hangi yıl ilan edildi?,1923,1923 | 29 Ekim 1923,1923.,1,1,NaN,NaN,NaN
2,Q03,Dünya'nın doğal uydusunun adı nedir?,Ay,Ay,Bilmiyorum.,0,0,NaN,NaN,NaN
3,Q04,Ay'a ilk ayak basan insan kimdir?,Neil Armstrong,Neil Armstrong | Neil Alden Armstrong,Neil Armstrong.,1,1,NaN,NaN,NaN
4,Q05,Suyun kimyasal formülü nedir?,H2O,H2O | H₂O,H₂O,1,1,NaN,NaN,NaN
5,Q06,İstiklal Marşı'nın yazarı kimdir?,Mehmet Akif Ersoy,Mehmet Akif Ersoy | Mehmet Âkif Ersoy,Mustafa Kemal Atatürk.,0,0,NaN,NaN,NaN
6,Q07,Bir üçgenin iç açılarının toplamı kaç derecedir?,180,180 | 180 derece,180 derece.,1,1,NaN,NaN,NaN
7,Q08,Güneş Sistemi'nin en büyük gezegeni hangisidir?,Jüpiter,Jüpiter,Jüpiter,1,1,NaN,NaN,NaN
8,Q09,Türkiye'nin en kalabalık şehri hangisidir?,İstanbul,İstanbul,İstanbul,1,1,NaN,NaN,NaN
9,Q10,DNA'nın Türkçe açılımı nedir?,Deoksiribonükleik asit,Deoksiribonükleik asit | Deoksiribonükleik asi...,DNA: Deoksiribonükleik Asit.,0,1,NaN,NaN,NaN


Etiketleme şablonu: /content/open_qa_pilot_60_annotation_template_v2.csv
CSV'yi indirip manual_is_correct, manual_error_type ve manual_notes alanlarını doldurun.


## İsteğe bağlı — Etiketleme CSV'sini Colab'dan indirme

In [ ]:
try:
    from google.colab import files
    files.download(
        str(ANNOTATION_TEMPLATE_PATH)
    )
except ImportError:
    print(
        "Bu hücre Google Colab dışında "
        "çalıştırılıyor."
    )

# Aşama 18 — Düzenlenmiş manuel etiketleri geri yükleme

Etiketlenmiş CSV'yi Colab'a yükledikten sonra dosya yolunu `UPLOADED_ANNOTATION_PATH` değişkenine yazın.

In [19]:
# Örnek:
# UPLOADED_ANNOTATION_PATH = "/content/open_qa_pilot_60_annotation_template_v2.csv"

UPLOADED_ANNOTATION_PATH = "/content/open_qa_pilot_60_annotation_manuel.csv"

if UPLOADED_ANNOTATION_PATH is None:
    print(
        "Etiketli CSV yolu henüz ayarlanmadı."
    )
else:

    labels_df = pd.read_csv(
                UPLOADED_ANNOTATION_PATH,
                sep=";",
                encoding="utf-8-sig",
                quotechar='"',
            )

    label_columns = [
        "question_id",
        "manual_is_correct",
        "manual_error_type",
        "manual_notes",
    ]

    merged_df = (
        raw_df.drop(
            columns=[
                "manual_is_correct",
                "manual_error_type",
                "manual_notes",
            ],
            errors="ignore",
        )
        .merge(
            labels_df[label_columns],
            on="question_id",
            how="left",
            validate="one_to_one",
        )
    )

    merged_df.to_csv(
        ANNOTATED_RESULTS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Etiketli sonuçlar:",
        ANNOTATED_RESULTS_PATH,
    )
    print(
        "Etiketlenen soru sayısı:",
        merged_df[
            "manual_is_correct"
        ].notna().sum(),
    )

Etiketli sonuçlar: /content/open_qa_pilot_60_annotated_v2.csv
Etiketlenen soru sayısı: 60


In [22]:
import pandas as pd

CSV_PATH = "/content/open_qa_pilot_60_annotation_manuel.csv"

manual_df = pd.read_csv(
    CSV_PATH,
    sep=";",
    encoding="utf-8-sig",
    quotechar='"',
)

print(manual_df.shape)
print(manual_df.columns.tolist())
manual_df.head()

(60, 10)
['question_id', 'question', 'reference_answer', 'accepted_aliases', 'generated_answer', 'automatic_exact_alias_match', 'automatic_contains_alias_match', 'manual_is_correct', 'manual_error_type', 'manual_notes']


,question_id,question,reference_answer,accepted_aliases,generated_answer,automatic_exact_alias_match,automatic_contains_alias_match,manual_is_correct,manual_error_type,manual_notes
0,Q01,Türkiye'nin başkenti neresidir?,Ankara,Ankara,Ankara.,1,1,1,NaN,NaN
1,Q02,Türkiye Cumhuriyeti hangi yıl ilan edildi?,1923,1923 | 29 Ekim 1923,1923.,1,1,1,NaN,NaN
2,Q03,Dünya'nın doğal uydusunun adı nedir?,Ay,Ay,Bilmiyorum.,0,0,0,NaN,NaN
3,Q04,Ay'a ilk ayak basan insan kimdir?,Neil Armstrong,Neil Armstrong | Neil Alden Armstrong,Neil Armstrong.,1,1,1,NaN,NaN
4,Q05,Suyun kimyasal formülü nedir?,H2O,H2O | H₂O,H₂O,1,1,1,NaN,NaN


# Aşama 19 — Leakage-safe post-hoc feature tablosu

## Çekirdek feature seti

- `mean_logprob`
- `min_logprob`
- `mean_token_entropy_normalized`
- `output_content_token_count`
- `self_confidence_scaled`
- `verification_p_correct`
- `self_consistency`
- `unknown_response`

## Genişletilmiş feature seti

Çekirdek sete ek olarak:

- `max_token_entropy_normalized`
- `first_token_entropy_normalized`
- `verification_p_uncertain`
- `verification_margin`
- `verification_entropy_normalized`
- `unique_answer_ratio`
- `sample_answer_entropy_normalized`
- `deterministic_majority_agreement`
- `unknown_sample_rate`
- `refusal_indicator`
- `refusal_sample_rate`

Aşağıdaki alanlar modele verilmez:

- `reference_answer`
- `accepted_aliases`
- `automatic_exact_alias_match`
- `automatic_contains_alias_match`
- `manual_error_type`
- sampled cevapların referansa göre doğruluk oranı

In [23]:
CORE_FEATURE_COLUMNS = [
    "mean_logprob",
    "min_logprob",
    "mean_token_entropy_normalized",
    "output_content_token_count",
    "self_confidence_scaled",
    "verification_p_correct",
    "self_consistency",
    "unknown_response",
]

EXTENDED_FEATURE_COLUMNS = [
    *CORE_FEATURE_COLUMNS,
    "max_token_entropy_normalized",
    "first_token_entropy_normalized",
    "verification_p_uncertain",
    "verification_margin",
    "verification_entropy_normalized",
    "unique_answer_ratio",
    "sample_answer_entropy_normalized",
    "deterministic_majority_agreement",
    "unknown_sample_rate",
    "refusal_indicator",
    "refusal_sample_rate",
]

if ANNOTATED_RESULTS_PATH.exists():
    modeling_df = pd.read_csv(
        ANNOTATED_RESULTS_PATH
    )
else:
    modeling_df = raw_df.copy()

model_feature_table = modeling_df[
    [
        "question_id",
        "model_id",
        *EXTENDED_FEATURE_COLUMNS,
        "manual_is_correct",
    ]
].copy()

model_feature_table.to_csv(
    MODEL_FEATURES_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(model_feature_table.head())

print(
    "Model feature tablosu:",
    MODEL_FEATURES_PATH,
)
print(
    "Çekirdek feature sayısı:",
    len(CORE_FEATURE_COLUMNS),
)
print(
    "Genişletilmiş feature sayısı:",
    len(EXTENDED_FEATURE_COLUMNS),
)

,question_id,model_id,mean_logprob,min_logprob,mean_token_entropy_normalized,output_content_token_count,self_confidence_scaled,verification_p_correct,self_consistency,unknown_response,...,verification_p_uncertain,verification_margin,verification_entropy_normalized,unique_answer_ratio,sample_answer_entropy_normalized,deterministic_majority_agreement,unknown_sample_rate,refusal_indicator,refusal_sample_rate,manual_is_correct
0,Q01,Qwen/Qwen3-4B,-0.014462,-0.055169,0.005395,4,1.00,1.000000e+00,1.0,0,...,8.864910e-10,1.000000,3.423174e-08,0.2,0.000000,1,0.0,0,0.0,1
1,Q02,Qwen/Qwen3-4B,-0.122272,-0.611336,0.020432,5,0.90,9.999993e-01,0.6,0,...,1.780564e-08,0.999999,1.011759e-05,0.4,0.418166,1,0.0,0,0.0,1
2,Q03,Qwen/Qwen3-4B,-0.000428,-0.001751,0.000362,6,0.50,6.796959e-10,1.0,1,...,7.112437e-07,-0.999999,9.825864e-06,0.2,0.000000,1,1.0,0,0.0,0
3,Q04,Qwen/Qwen3-4B,-0.035664,-0.097295,0.010418,3,0.85,1.000000e+00,1.0,0,...,1.323203e-08,1.000000,4.939313e-07,0.2,0.000000,1,0.0,0,0.0,1
4,Q05,Qwen/Qwen3-4B,-0.001190,-0.003552,0.000666,3,0.95,1.000000e+00,1.0,0,...,3.638152e-10,1.000000,1.025213e-08,0.2,0.000000,1,0.0,0,0.0,1


Model feature tablosu: /content/open_qa_pilot_60_model_features_v2.csv
Çekirdek feature sayısı: 8
Genişletilmiş feature sayısı: 19


# Aşama 20 — Pilot düzeyinde post-hoc fusion baseline

Bu aşama yalnızca bütün `manual_is_correct` etiketleri doldurulduktan sonra çalıştırılmalıdır.

60 soru, genişletilmiş feature setiyle güvenilir bir nihai model eğitmek için küçüktür. Bu nedenle pilot aşamasında yalnızca:

- Pipeline'ın çalıştığını,
- Feature'ların parse edildiğini,
- Tek sinyal ve çoklu sinyal eğilimlerini

kontrol etmek amaçlanır.

Nihai tez deneyinde daha büyük calibration/test seti ve bootstrap confidence interval kullanılmalıdır.

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def expected_calibration_error(
    y_true,
    probabilities,
    n_bins: int = 10,
) -> float:
    y_true = np.asarray(y_true)
    probabilities = np.asarray(
        probabilities
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for lower, upper in zip(
        bin_edges[:-1],
        bin_edges[1:],
    ):
        if upper == 1.0:
            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        if not mask.any():
            continue

        bin_accuracy = y_true[mask].mean()
        bin_confidence = (
            probabilities[mask].mean()
        )

        ece += (
            mask.mean()
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


labeled_df = model_feature_table.dropna(
    subset=["manual_is_correct"]
).copy()

if len(labeled_df) < 30:
    print(
        "En az 30 manuel etiketli örnek "
        "olmadan pilot fusion çalıştırılmadı."
    )
else:
    labeled_df[
        "manual_is_correct"
    ] = labeled_df[
        "manual_is_correct"
    ].astype(int)

    class_counts = labeled_df[
        "manual_is_correct"
    ].value_counts()

    if len(class_counts) < 2:
        print(
            "Doğru ve yanlış olmak üzere "
            "iki sınıf da gerekli."
        )
    else:
        min_class_count = int(
            class_counts.min()
        )
        n_splits = min(
            5,
            min_class_count,
        )

        if n_splits < 2:
            print(
                "Cross-validation için "
                "yeterli azınlık sınıfı yok."
            )
        else:
            X = labeled_df[
                CORE_FEATURE_COLUMNS
            ]
            y = labeled_df[
                "manual_is_correct"
            ]

            estimator = Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        ),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                    (
                        "model",
                        LogisticRegression(
                            penalty="l2",
                            max_iter=2000,
                            class_weight="balanced",
                            random_state=42,
                        ),
                    ),
                ]
            )

            cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=42,
            )

            fused_probabilities = (
                cross_val_predict(
                    estimator,
                    X,
                    y,
                    cv=cv,
                    method="predict_proba",
                )[:, 1]
            )

            fused_predictions = (
                fused_probabilities >= 0.5
            ).astype(int)

            metrics = {
                "n": len(y),
                "accuracy": accuracy_score(
                    y,
                    fused_predictions,
                ),
                "brier": brier_score_loss(
                    y,
                    fused_probabilities,
                ),
                "nll": log_loss(
                    y,
                    np.clip(
                        fused_probabilities,
                        1e-6,
                        1 - 1e-6,
                    ),
                ),
                "auroc": roc_auc_score(
                    y,
                    fused_probabilities,
                ),
                "ece_10_bins_exploratory": (
                    expected_calibration_error(
                        y,
                        fused_probabilities,
                        n_bins=10,
                    )
                ),
                "mean_predicted_confidence": (
                    fused_probabilities.mean()
                ),
                "empirical_accuracy": (
                    y.mean()
                ),
                "overconfidence": (
                    fused_probabilities.mean()
                    - y.mean()
                ),
            }

            display(
                pd.DataFrame(
                    [metrics]
                )
            )

,n,accuracy,brier,nll,auroc,ece_10_bins_exploratory,mean_predicted_confidence,empirical_accuracy,overconfidence
0,60,0.933333,0.075739,0.283578,0.93875,0.086763,0.610547,0.666667,-0.05612


# Aşama 21 — Feature kalite kontrol raporu

In [25]:
feature_quality_rows = []

for feature in EXTENDED_FEATURE_COLUMNS:
    series = pd.to_numeric(
        model_feature_table[feature],
        errors="coerce",
    )

    feature_quality_rows.append(
        {
            "feature": feature,
            "missing_count": int(
                series.isna().sum()
            ),
            "missing_rate": float(
                series.isna().mean()
            ),
            "unique_count": int(
                series.nunique(
                    dropna=True
                )
            ),
            "mean": float(
                series.mean()
            ) if series.notna().any() else np.nan,
            "std": float(
                series.std()
            ) if series.notna().sum() > 1 else np.nan,
            "min": float(
                series.min()
            ) if series.notna().any() else np.nan,
            "max": float(
                series.max()
            ) if series.notna().any() else np.nan,
        }
    )

feature_quality_df = pd.DataFrame(
    feature_quality_rows
)

display(feature_quality_df)

print(
    "Sabit veya varyansı çok düşük "
    "feature'lar nihai modelden "
    "çıkarılmalıdır."
)

,feature,missing_count,missing_rate,unique_count,mean,std,min,max
0,mean_logprob,0,0.0,60,-0.095825,0.112672,-4.666646e-01,-0.000002
1,min_logprob,0,0.0,60,-0.386482,0.413826,-2.149941e+00,-0.000002
2,mean_token_entropy_normalized,0,0.0,60,0.019981,0.022284,2.051218e-06,0.102836
3,output_content_token_count,0,0.0,12,5.833333,3.247772,1.000000e+00,20.000000
4,self_confidence_scaled,0,0.0,12,0.744000,0.227940,0.000000e+00,1.000000
5,verification_p_correct,0,0.0,60,0.574902,0.488418,2.084827e-11,1.000000
6,self_consistency,0,0.0,4,0.830000,0.217302,4.000000e-01,1.000000
7,unknown_response,0,0.0,2,0.200000,0.403376,0.000000e+00,1.000000
8,max_token_entropy_normalized,0,0.0,60,0.071118,0.072197,2.877641e-06,0.387863
9,first_token_entropy_normalized,0,0.0,60,0.033177,0.045343,1.369468e-07,0.194376


Sabit veya varyansı çok düşük feature'lar nihai modelden çıkarılmalıdır.


# Aşama 22 — Çıktı dosyaları

Notebook tamamlandığında aşağıdaki dosyalar oluşur:

- `open_qa_pilot_60_raw_v2.csv`  
  Ham üretimler, bütün sinyaller, metadata ve otomatik ön kontroller.

- `open_qa_pilot_60_annotation_template_v2.csv`  
  Manuel doğruluk ve hata türü etiketleme şablonu.

- `open_qa_pilot_60_annotated_v2.csv`  
  Manuel etiketler ham sonuçlarla birleştirildikten sonra oluşur.

- `open_qa_pilot_60_model_features_v2.csv`  
  Post-hoc model için leakage-safe feature matrisi ve hedef sütunu.

## Sonraki adım

Pilot sonuçları tamamlandıktan sonra:

1. Feature dağılımlarını doğru/yanlış gruplarında karşılaştırın.
2. Tek sinyal baseline'larını kurun.
3. Çoklu sinyal logistic fusion modelini değerlendirin.
4. Daha büyük bir calibration/test veri setine geçin.
5. Nihai tez deneyinde bootstrap güven aralıkları ve selective answering analizi ekleyin.